In [1]:
"""
Climate Indices: Download and Update (AO, NAO, PDO, PNA, SCAND, Nino4, GBI, IPO-TPI)
========================================================================================

Downloads current, complete versions of the NOAA/PSL monthly climate indices
already in use (AO, NAO, PDO, PNA, SCAND), plus three new ones:
  - Nino 4 SST anomaly       -- input for the TPQD index (Xu et al. 2026 method)
  - Greenland Blocking Index -- GBI (UL), University of Lincoln version
  - IPO Tripole Index        -- Henley et al. (2015)

Also computes the TPQD (Tropical Pacific Quasi-Decadal) index following
Xu et al. (2026)'s own definition: 9-16yr Butterworth bandpass filtered
Nino 4 SST. TPQD is not distributed as a ready-made product anywhere;
this reconstructs it directly, reusing the identical bandpass function
from figure_s06_QD_cycle_SSA_extraction.ipynb for consistency.

IMPORTANT -- URL verification status
-------------------------------------
Source URLs below were identified via web search of PSL's site
(https://psl.noaa.gov/data/timeseries/month/ and legacy
https://psl.noaa.gov/data/correlation/) rather than from within this
notebook's own execution environment (psl.noaa.gov is not reachable from
the sandbox this was drafted in). Confidence varies by index:

  HIGH CONFIDENCE  (long-standing, stable legacy PSL paths, independently
                    corroborated by external citations):
      AO, NAO, PDO, PNA, SCAND, Nino4

  CONFIRMED DIRECT (fetched and read the actual download page):
      IPO Tripole Index (filtered + unfiltered, ERSSTv5)

  CONFIRMED DIRECT:
      GBI (UL), downloaded automatically from
      https://psl.noaa.gov/data/timeseries/month/data/gbi.mon.csv. Its
      format differs from the other indices -- long-format Date/value
      rows, not the PSL wide format -- so it has its own dedicated parser
      (parse_gbi_ul_csv, Section 1b) rather than going through
      parse_psl_standard_format. Parser verified against a manually-
      obtained copy of this same file before the URL was confirmed.

All other indices go through the same generic PSL-standard-format parser
(parse_psl_standard_format), so once a correct URL is in place for any of
those, no other code needs to change.

Outputs
-------
  Raw downloads:   ../../climate_indices/raw/{name}.data (or .csv for GBI)
  Processed CSVs:  ./outputs/processed_data/climate_indices/{name}.csv
  Merged wide CSV: ./outputs/processed_data/climate_indices/all_indices_merged.csv
  TPQD CSV:        ./outputs/processed_data/climate_indices/TPQD_index.csv
  Combined NetCDF: ./outputs/processed_data/climate_indices/all_climate_indices.nc
                   (all indices + TPQD on one time grid, one variable each,
                   per-variable long_name/source attrs -- the recommended
                   single file for downstream analysis notebooks)
  Methodology:     ./outputs/methods/climate_indices_methodology.md
  Log:             ./outputs/logs/climate_indices.log

Version:       0.1.0 (draft -- GBI (UL) URL needs manual verification)
Last Modified: 2026-08-19
Author:        Chris Barrell
"""

import os
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import requests
from scipy import signal

warnings.filterwarnings("ignore")

sys.path.append('..')
try:
    from utils.logger import get_logger
    logger = get_logger(__name__, config_path='../config.yaml')
except Exception:
    logger = logging.getLogger(__name__)
    logging.basicConfig(level=logging.INFO)

# ============================================================================
# CONFIGURATION
# ============================================================================

RAW_DIR       = Path('../../climate_indices/raw/')
PROCESSED_DIR = Path('./outputs/processed_data/climate_indices/')
METHODS_DIR   = Path('./outputs/methods/')
LOG_DIR       = Path('./outputs/logs/')
for d in [RAW_DIR, PROCESSED_DIR, METHODS_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log_path = LOG_DIR / 'climate_indices.log'
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
logger.addHandler(fh)

REPROCESS = True   # Set False after first successful run to reuse cached downloads

# --- GBI (UL): confirmed direct download URL (University of Lincoln version).
# Format differs from the PSL standard format used by the other indices --
# long-format Date/value CSV, not wide Year+12-months -- so it has its own
# parser (parse_gbi_ul_csv, Section 1b) rather than parse_psl_standard_format.
GBI_UL_URL       = 'https://psl.noaa.gov/data/timeseries/month/data/gbi.mon.csv'
GBI_UL_LOCAL_CSV = RAW_DIR / 'gbi_mon.csv'  # download target / cache; also used as a
                                             # manual fallback if the URL ever moves

# --- Index registry: name -> (url, description) ---
INDEX_SOURCES = {
    # -- Already in use, being refreshed --
    'AO':    ('https://psl.noaa.gov/data/correlation/ao.data',
              'Arctic Oscillation, NOAA/CPC, 1st EOF of 1000mb heights N of 20N'),
    'NAO':   ('https://psl.noaa.gov/data/correlation/nao.data',
              'North Atlantic Oscillation, NOAA/NGDC, Azores-Iceland SLP difference'),
    'PDO':   ('https://psl.noaa.gov/data/correlation/pdo.data',
              'Pacific Decadal Oscillation, U of Washington, leading EOF N Pacific SSTA'),
    'PNA':   ('https://psl.noaa.gov/data/correlation/pna.data',
              'Pacific North American pattern, NOAA/CPC'),
    'SCAND': ('https://psl.noaa.gov/data/correlation/scand.data',
              'Scandinavian pattern, NOAA/CPC, blocking over W Russia/Scandinavia'),

    # -- New: Nino 4 for TPQD reconstruction (Xu et al. 2026 method) --
    'NINO4': ('https://psl.noaa.gov/data/correlation/nina4.data',
              'Nino 4 SST anomaly, 5N-5S 160E-150W -- input for TPQD (NOT the same '
              'region as Nino 3.4; do not substitute)'),
    'NINO34': ('https://psl.noaa.gov/data/correlation/nina34.data',
               'Nino 3.4 SST anomaly, 5N-5S 170W-120W -- standard ENSO index, for direct '
               'interannual (2-7yr band) teleconnection testing, distinct from TPQD'),

    # -- New: IPO Tripole Index (Henley et al. 2015), confirmed direct URL --
    'IPO_TPI_FILT': ('https://psl.noaa.gov/data/timeseries/IPOTPI/tpi.timeseries.ersstv5.filt.data',
                      'IPO Tripole Index, low-pass filtered, ERSSTv5 (Henley et al. 2015) '
                      '-- this is the standard, commonly-cited version'),
    'IPO_TPI_UNFILT': ('https://psl.noaa.gov/data/timeseries/IPOTPI/tpi.timeseries.ersstv5.data',
                        'IPO Tripole Index, unfiltered, ERSSTv5 (Henley et al. 2015)'),
}

# NOTE: GBI (UL) is NOT added to INDEX_SOURCES -- it doesn't use the PSL
# standard format the generic downloader/parser expects, so it's handled
# separately in Section 1b / Section 2 via parse_gbi_ul_csv().

# --- TPQD bandpass parameters (identical to figure_s06) ---
TPQD_LOW_YR  = 9
TPQD_HIGH_YR = 16


## Section 1 - Generic PSL standard-format downloader/parser

In [2]:
def download_raw(name, url, raw_dir=RAW_DIR, reprocess=REPROCESS, timeout=30, ext='.data'):
    """Download a PSL-format .data file to disk, or reuse a cached copy."""
    raw_path = raw_dir / f'{name}{ext}'
    if not reprocess and raw_path.exists():
        logger.info(f"[{name}] Using cached raw file: {raw_path}")
        return raw_path

    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        logger.error(f"[{name}] Download FAILED from {url}: {e}")
        logger.error(f"  -> Check the URL manually at https://psl.noaa.gov/data/timeseries/month/ "
                     f"and update its source URL (INDEX_SOURCES['{name}'] for the PSL-format "
                     f"indices, or GBI_UL_URL for GBI) if it has moved.")
        return None

    raw_path.write_text(resp.text)
    logger.info(f"[{name}] Downloaded {len(resp.text)} bytes -> {raw_path}")
    return raw_path


def parse_psl_standard_format(raw_path, name):
    """Parse a PSL 'standard format' monthly index file:

        start_year end_year
        year1 jan feb mar apr may jun jul aug sep oct nov dec
        year2 jan feb mar ...
        ...
        yearN jan feb mar ...
        missing_value
        <citation / free-text lines, ignored>

    Returns a tidy DataFrame with columns: Year, Jan, ..., Dec (missing
    values replaced with NaN), plus the detected missing-value sentinel
    for reference.
    """
    if raw_path is None:
        return None, None

    lines = raw_path.read_text().splitlines()
    lines = [ln.strip() for ln in lines if ln.strip()]

    if not lines:
        logger.error(f"[{name}] Raw file is empty: {raw_path}")
        return None, None

    # First line: start_year end_year
    header_tokens = lines[0].split()
    if len(header_tokens) < 2:
        logger.error(f"[{name}] Unexpected header line, not 'start_year end_year': "
                     f"'{lines[0]}'. File may not be in PSL standard format -- "
                     f"inspect {raw_path} manually.")
        return None, None
    start_year, end_year = int(float(header_tokens[0])), int(float(header_tokens[1]))
    n_years_expected = end_year - start_year + 1

    rows = []
    i = 1
    while i < len(lines) and len(rows) < n_years_expected:
        tokens = lines[i].split()
        if len(tokens) == 13:
            try:
                year = int(float(tokens[0]))
                vals = [float(t) for t in tokens[1:]]
                rows.append([year] + vals)
            except ValueError:
                break  # hit a non-data line
        else:
            break
        i += 1

    if not rows:
        logger.error(f"[{name}] No data rows parsed from {raw_path} -- format may have "
                     f"changed. First few lines: {lines[:3]}")
        return None, None

    df = pd.DataFrame(rows, columns=['Year', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                                       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

    # Next non-data line is typically the missing-value sentinel
    missing_value = None
    if i < len(lines):
        try:
            missing_value = float(lines[i].split()[0])
        except (ValueError, IndexError):
            pass

    if missing_value is not None:
        month_cols = df.columns[1:]
        df[month_cols] = df[month_cols].mask(np.isclose(df[month_cols], missing_value))
    else:
        logger.warning(f"[{name}] Could not detect missing-value sentinel after data "
                        f"block -- large sentinel values (e.g. -999, -99.9, 99.99) may "
                        f"remain in the output uncleaned. Inspect manually.")

    logger.info(f"[{name}] Parsed {len(df)} years ({df['Year'].min():.0f}-"
                f"{df['Year'].max():.0f}), missing_value={missing_value}")
    return df, missing_value


## Section 1b - GBI (UL) parser: different format from the others (long-format Date/value CSV, not PSL wide format)

In [3]:
MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

def parse_gbi_ul_csv(path, name='GBI_UL'):
    """Parse the University of Lincoln GBI file (gbi_mon.csv).

    Format: 'Date, value' long-format CSV with a single messy header row,
    e.g. '1851-01-01,   -0.900'. NOT the PSL standard wide format used by
    the other indices.

    Missing-value handling: the header text claims '-99' is the missing
    flag, but the actual trailing rows use -9999.000 -- both conventions
    are masked defensively here (anything with |value| >= 99), since -99
    exactly and -9999 both occur in practice and neither is a plausible
    real GBI value (a standardized/anomaly-like index, values in this file
    range roughly -4 to +4).

    Returns a wide DataFrame (Year, Jan..Dec) matching the same shape as
    parse_psl_standard_format's output, so it flows through the existing
    to_long()/merge pipeline unchanged.
    """
    if path is None or not path.exists():
        logger.error(f"[{name}] File not found: {path}. Check GBI_UL_URL is reachable, "
                     f"or manually copy a gbi_mon.csv to GBI_UL_LOCAL_CSV.")
        return None

    df = pd.read_csv(path, skiprows=1, header=None, names=['Date', 'value'])
    df['Date'] = pd.to_datetime(df['Date'].astype(str).str.strip())
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

    n_before = df['value'].notna().sum()
    df.loc[df['value'].abs() >= 99, 'value'] = np.nan
    n_masked = n_before - df['value'].notna().sum()
    logger.info(f"[{name}] Masked {n_masked} sentinel/missing values "
                f"(|value| >= 99, covers both the documented -99 and the "
                f"actual -9999.000 used in the file)")

    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month

    wide = df.pivot(index='Year', columns='Month', values='value')
    wide = wide.reindex(columns=range(1, 13))  # ensure all 12 months present
    wide.columns = MONTH_NAMES
    wide = wide.reset_index()

    logger.info(f"[{name}] Parsed {len(wide)} years ({wide['Year'].min():.0f}-"
                f"{wide['Year'].max():.0f}) from {path}")
    return wide


## Section 2 - Download and parse all indices

In [4]:
parsed_indices = {}

# GBI (UL) -- handled separately, different file format (see Section 1b)
gbi_raw_path = download_raw('gbi_mon', GBI_UL_URL, ext='.csv')
if gbi_raw_path is None and GBI_UL_LOCAL_CSV.exists():
    logger.warning("[GBI_UL] Download failed -- falling back to existing local file "
                    f"at {GBI_UL_LOCAL_CSV}")
    gbi_raw_path = GBI_UL_LOCAL_CSV
gbi_df = parse_gbi_ul_csv(gbi_raw_path)
if gbi_df is not None:
    out_csv = PROCESSED_DIR / 'GBI_UL.csv'
    gbi_df.to_csv(out_csv, index=False)
    logger.info(f"[GBI_UL] Saved: {out_csv}")
    parsed_indices['GBI_UL'] = gbi_df
else:
    logger.warning("[GBI_UL] SKIPPED -- see errors above.")

for name, (url, description) in INDEX_SOURCES.items():
    logger.info(f"--- {name}: {description}")
    raw_path = download_raw(name, url)
    df, missing_value = parse_psl_standard_format(raw_path, name)
    if df is not None:
        out_csv = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(out_csv, index=False)
        logger.info(f"[{name}] Saved: {out_csv}")
        parsed_indices[name] = df
    else:
        logger.warning(f"[{name}] SKIPPED -- see errors above.")

logger.info(f"Successfully parsed {len(parsed_indices)} / {len(INDEX_SOURCES)} indices: "
            f"{list(parsed_indices.keys())}")


INFO:__main__:[gbi_mon] Downloaded 44196 bytes -> ../../climate_indices/raw/gbi_mon.csv
INFO:__main__:[GBI_UL] Masked 4 sentinel/missing values (|value| >= 99, covers both the documented -99 and the actual -9999.000 used in the file)
INFO:__main__:[GBI_UL] Parsed 175 years (1851-2025) from ../../climate_indices/raw/gbi_mon.csv
INFO:__main__:[GBI_UL] Saved: outputs/processed_data/climate_indices/GBI_UL.csv
INFO:__main__:--- AO: Arctic Oscillation, NOAA/CPC, 1st EOF of 1000mb heights N of 20N
INFO:__main__:[AO] Downloaded 8922 bytes -> ../../climate_indices/raw/AO.data
INFO:__main__:[AO] Parsed 77 years (1950-2026), missing_value=-999.0
INFO:__main__:[AO] Saved: outputs/processed_data/climate_indices/AO.csv
INFO:__main__:--- NAO: North Atlantic Oscillation, NOAA/NGDC, Azores-Iceland SLP difference
INFO:__main__:[NAO] Downloaded 7218 bytes -> ../../climate_indices/raw/NAO.data
INFO:__main__:[NAO] Parsed 79 years (1948-2026), missing_value=-99.9
INFO:__main__:[NAO] Saved: outputs/processed

## Section 3 - Long-format helper + merged wide CSV (all indices, common Year/Month grid)

In [5]:
def to_long(df, name):
    """Wide (Year, Jan..Dec) -> long (Year, Month, <name>) format."""
    month_map = {m: i + 1 for i, m in enumerate(
        ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])}
    long_df = df.melt(id_vars='Year', var_name='MonthName', value_name=name)
    long_df['Month'] = long_df['MonthName'].map(month_map)
    long_df['time'] = pd.to_datetime(dict(year=long_df['Year'], month=long_df['Month'], day=1))
    return long_df[['time', name]].sort_values('time').reset_index(drop=True)


long_series = {name: to_long(df, name) for name, df in parsed_indices.items()}

merged = None
for name, s in long_series.items():
    merged = s if merged is None else merged.merge(s, on='time', how='outer')

if merged is not None:
    merged = merged.sort_values('time').reset_index(drop=True)
    merged_csv = PROCESSED_DIR / 'all_indices_merged.csv'
    merged.to_csv(merged_csv, index=False)
    logger.info(f"Merged wide-format CSV saved: {merged_csv} "
                f"({merged['time'].min().date()} to {merged['time'].max().date()})")
merged


INFO:__main__:Merged wide-format CSV saved: outputs/processed_data/climate_indices/all_indices_merged.csv (1851-01-01 to 2026-12-01)


,time,GBI_UL,AO,NAO,PDO,PNA,SCAND,NINO4,NINO34,IPO_TPI_FILT,IPO_TPI_UNFILT
0,1851-01-01,-0.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1851-02-01,-0.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1851-03-01,-0.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1851-04-01,-0.02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1851-05-01,0.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2107,2026-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2108,2026-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2109,2026-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2110,2026-11-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Section 4 - TPQD index: 9-16yr Butterworth bandpass on Nino 4 (Xu et al. 2026 method)

In [6]:
def butterworth_bandpass_extract(y, low_yr=TPQD_LOW_YR, high_yr=TPQD_HIGH_YR,
                                  fs=12.0, order=4):
    """9-16yr Butterworth bandpass on a MONTHLY series (fs=12 samples/yr).
    Identical method to figure_s06_QD_cycle_SSA_extraction.ipynb's
    butterworth_bandpass_extract, adapted here for monthly (not annual)
    input, matching Xu et al. (2026)'s TPQD definition exactly:
    'TPQD index, defined by 9-16-year bandpass filtered Nino4 SST anomalies'.
    """
    nyq = 0.5 * fs
    low_freq  = (1.0 / high_yr) / nyq
    high_freq = (1.0 / low_yr) / nyq
    b, a = signal.butter(order, [low_freq, high_freq], btype='band')
    pad = min(len(y) - 1, 3 * order * 2 * 12)  # scale padding for monthly resolution
    y_padded = np.concatenate([y[pad:0:-1], y, y[-2:-pad - 2:-1]])
    filtered = signal.filtfilt(b, a, y_padded)
    return filtered[pad:pad + len(y)]


if 'NINO4' in long_series:
    nino4 = long_series['NINO4'].dropna(subset=['NINO4']).reset_index(drop=True)
    tpqd_values = butterworth_bandpass_extract(nino4['NINO4'].values)
    tpqd_df = pd.DataFrame({'time': nino4['time'], 'TPQD': tpqd_values})
    tpqd_csv = PROCESSED_DIR / 'TPQD_index.csv'
    tpqd_df.to_csv(tpqd_csv, index=False)
    logger.info(f"TPQD index computed and saved: {tpqd_csv} "
                f"({len(tpqd_df)} months, variance fraction of Nino4 retained: "
                f"{np.var(tpqd_values) / np.var(nino4['NINO4'].values):.3f})")
else:
    logger.warning("NINO4 not available -- TPQD index not computed. Check the NINO4 "
                    "download above.")
    tpqd_df = None

tpqd_df


INFO:__main__:TPQD index computed and saved: outputs/processed_data/climate_indices/TPQD_index.csv (943 months, variance fraction of Nino4 retained: 0.146)


,time,TPQD
0,1948-01-01,-0.028374
1,1948-02-01,-0.027422
2,1948-03-01,-0.026539
3,1948-04-01,-0.025724
4,1948-05-01,-0.024979
...,...,...
938,2026-03-01,0.305043
939,2026-04-01,0.308833
940,2026-05-01,0.311835
941,2026-06-01,0.314047


## Section 6 - Methodology writeup

## Section 5 - Combine all indices + TPQD into a single NetCDF

In [8]:
# Descriptions for variable attrs (reuses INDEX_SOURCES + the two indices
# that don't come from that dict: GBI_UL and the derived TPQD index)
VAR_DESCRIPTIONS = {name: desc for name, (url, desc) in INDEX_SOURCES.items()}
VAR_DESCRIPTIONS['GBI_UL'] = ('Greenland Blocking Index (UL), mean 500hPa geopotential '
                               'height over 60-80N, 20-80W (University of Lincoln)')
VAR_DESCRIPTIONS['TPQD'] = (f'Tropical Pacific Quasi-Decadal index: {TPQD_LOW_YR}-{TPQD_HIGH_YR}yr '
                             f'Butterworth bandpass of Nino4 SST anomaly, following Xu et al. (2026)')

VAR_SOURCES = {name: url for name, (url, desc) in INDEX_SOURCES.items()}
VAR_SOURCES['GBI_UL'] = GBI_UL_URL
VAR_SOURCES['TPQD'] = 'derived (this notebook), from NINO4'


def build_combined_dataset(merged_df, tpqd_df):
    """Combine the merged wide-format index table and the derived TPQD
    series onto a single time grid, return as an xarray Dataset with
    per-variable metadata."""
    if merged_df is None:
        logger.error("No merged index data available -- run Section 2/3 first.")
        return None

    combined = merged_df.copy()
    if tpqd_df is not None:
        combined = combined.merge(tpqd_df, on='time', how='outer')

    combined = combined.sort_values('time').reset_index(drop=True)
    ds = combined.set_index('time').to_xarray()

    for var in ds.data_vars:
        ds[var].attrs['long_name'] = VAR_DESCRIPTIONS.get(var, '')
        ds[var].attrs['source'] = VAR_SOURCES.get(var, '')

    ds.attrs['title'] = 'Combined climate indices for Greenland Sea / ROVER analysis'
    ds.attrs['history'] = 'Created by climate_indices_download.ipynb'
    ds.attrs['note'] = ('Index definitions, source datasets, and climatology periods '
                        'differ between variables -- see per-variable "source" attribute '
                        'and climate_indices_methodology.md before combining/comparing '
                        'across indices.')
    return ds


combined_ds = build_combined_dataset(merged, tpqd_df)

if combined_ds is not None:
    nc_path = PROCESSED_DIR / 'all_climate_indices.nc'
    # Use a fixed encoding so re-running doesn't silently change dtypes
    encoding = {var: {'zlib': True, 'complevel': 4} for var in combined_ds.data_vars}
    combined_ds.to_netcdf(nc_path, encoding=encoding)
    logger.info(f"Combined NetCDF saved: {nc_path} "
                f"({len(combined_ds.data_vars)} variables, "
                f"{combined_ds.time.min().values} to {combined_ds.time.max().values})")

combined_ds


INFO:__main__:Combined NetCDF saved: outputs/processed_data/climate_indices/all_climate_indices.nc (11 variables, 1851-01-01T00:00:00.000000000 to 2026-12-01T00:00:00.000000000)


<xarray.Dataset> Size: 203kB
Dimensions:         (time: 2112)
Coordinates:
  * time            (time) datetime64[ns] 17kB 1851-01-01 ... 2026-12-01
Data variables:
    GBI_UL          (time) float64 17kB -0.9 -0.6 -0.51 -0.02 ... nan nan nan
    AO              (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    NAO             (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    PDO             (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    PNA             (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    SCAND           (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    NINO4           (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    NINO34          (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    IPO_TPI_FILT    (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    IPO_TPI_UNFILT  (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
    TPQD            (time) float64 17kB nan nan nan nan nan ... nan nan nan nan
Attributes:
    title:    Combined climate indices for Greenland Sea / ROVER analysis
    history:  Created by climate_indices_download.ipynb
    note:     Index definitions, source datasets, and climatology periods dif...

In [ ]:
def write_methodology():
    lines = [
        "# Climate Indices: Sources and Processing",
        "",
        "## Indices downloaded",
    ]
    for name, (url, desc) in INDEX_SOURCES.items():
        status = "parsed OK" if name in parsed_indices else "FAILED / SKIPPED"
        lines.append(f"- **{name}** [{status}]: {desc}")
        lines.append(f"  Source: {url}")

    gbi_status = "parsed OK" if 'GBI_UL' in parsed_indices else "FAILED / SKIPPED"
    lines.append(f"- **GBI_UL** [{gbi_status}]: Greenland Blocking Index (UL), mean "
                 f"500hPa Z over 60-80N 20-80W")
    lines.append(f"  Source: {GBI_UL_URL} "
                 f"(University of Lincoln; cached at {GBI_UL_LOCAL_CSV})")
    lines += [
        "",
        "## TPQD (Tropical Pacific Quasi-Decadal) index",
        "Not a distributed product anywhere; reconstructed here following Xu et al.",
        f"(2026)'s definition: {TPQD_LOW_YR}-{TPQD_HIGH_YR}yr Butterworth bandpass of",
        "Nino 4 SST anomaly (5N-5S, 160E-150W). Note this uses Nino 4, NOT Nino 3.4 --",
        "do not substitute the existing Nino34 file for this purpose.",
        "",
        "## Known caveats",
        "- GBI (UL) uses a different file format from the other indices",
        "  (long-format Date/value CSV, not PSL wide format). Missing values",
        "  masked defensively at |value| >= 99 (covers both the documented -99 sentinel",
        "  and the -9999.000 actually used in the file's trailing rows).",
        "- IPO Tripole Index: filtered (IPO_TPI_FILT) is the standard, commonly-cited",
        "  version (Henley et al. 2015); unfiltered version also saved for reference.",
        "- PSL notes that even for the same phenomenon, index definitions/datasets/",
        "  climatologies can differ between sources -- these are not necessarily",
        "  interchangeable with indices computed elsewhere under the same name.",
    ]
    with open(METHODS_DIR / 'climate_indices_methodology.md', 'w') as f:
        f.write('\n'.join(lines))
    logger.info(f"Methodology written: {METHODS_DIR / 'climate_indices_methodology.md'}")

write_methodology()
